# The Brownian-lineage per-family rate layer: retire it
36 vendor rates collapse to one rate (spread across vendors 0.01–0.04 logits/yr against 0.49–0.65 posterior width).
It cost 5.9× sampling time and a second posterior mode: 10,064 s vs 1,712 s, identified eta r̂ 1.272 vs 1.003.
Kept: the per-axis drift with `dt` scaling, which measures a real 3.5× spread in capability climb rates.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "fit.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import xarray as xr
import arviz as az
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde

from data import load_eci_data
from lineage import build_lineage_structure
from analysis import mirt_identified_rhat

FIT = "mirt_humanprior_lineageprior_lineagebm_floors_ratehier"
TRACE = ROOT / f"results/{FIT}/trace_mirt_k3_ratehier.nc"
PLOTS = ROOT / "plots/lineage_bm_rate_diagnosis"
PLOTS.mkdir(parents=True, exist_ok=True)
THIN = 5

C = dict(blue="#0072B2", sky="#56B4E9", orange="#E69F00", verm="#D55E00",
         green="#009E73", pink="#CC79A7", gray="#999999", dark="#333333")

def show(fig, name):
    fig.update_layout(template="plotly_white", title=dict(x=0.5, font=dict(size=13)))
    fig.write_image(PLOTS / f"{name}.png", scale=2)
    fig.show()

# ── trace: only the variables the claims need, thinned ───────────────────────
VARS = ["A", "theta", "D", "sigma_b", "tau_CD", "lin_rate", "lin_rate_mu", "lin_rate_sd"]
post = xr.open_dataset(TRACE, group="posterior")[VARS].isel(draw=slice(None, None, THIN)).load()
stats = xr.open_dataset(TRACE, group="sample_stats")[["logp", "n_steps", "diverging"]].load()

data = load_eci_data(include_all_benchmarks=True)
lin = build_lineage_structure(data.mlookup)
benches = data.blookup.sort_values("benchmark_idx")["benchmark"].tolist()

# ── axis relabelling: best permutation of each chain onto chain 0 ────────────
A = post["A"].values                       # (chain, draw, bench, latent)
A0 = np.median(A[0], axis=0)

def perm_to0(c):
    Ac = np.median(A[c], axis=0)
    M = np.array([[np.corrcoef(A0[:, i], Ac[:, j])[0, 1] for j in range(3)] for i in range(3)])
    return M.argmax(axis=1), M.max(axis=1)

PERM, CORR = map(np.array, zip(*[perm_to0(c) for c in range(8)]))
HEALTHY, ODD = list(range(7)), 7

def aligned(var, chains):
    """Draws of `var` pooled over `chains`, axes relabelled onto the chain-0 frame."""
    x = post[var].values
    return np.concatenate([x[c][..., PERM[c]] for c in chains], axis=0)

# ── axis identity, read off the consensus loadings ───────────────────────────
A_cons = np.median(aligned("A", HEALTHY), axis=0)
A_odd = np.median(A[ODD][..., PERM[ODD]], axis=0)
NAME = {"FrontierMath Tier 4": "Hard math + science",     # KeyError = axes moved
        "VPCT": "Fluid/abstract + agentic",
        "GBAEval": "Easy knowledge"}
def top_benches(k, n=3):
    return [benches[i] for i in np.argsort(-A_cons[:, k])[:n]]
AXIS = [NAME[top_benches(k, 1)[0]] for k in range(3)]
LEG = [f"{AXIS[k]} ({', '.join(top_benches(k))})" for k in range(3)]
EASY, FLUID = AXIS.index("Easy knowledge"), AXIS.index("Fluid/abstract + agentic")

# 12 benchmarks the 2026-07-27 refresh added (85 -> 97), all thinly evaluated
NEW = {"AlgoTune", "BTF3", "BlueprintBench 2", "DeepSWE", "EBR-bench", "GBAEval",
       "GDP.pdf", "GDPval", "MindCube", "ProofBench", "SpatialViz-Bench", "Surface Evolver Bench"}

print(f"{post.sizes['draw']} draws/chain · {len(data.scores)} obs · {data.n_models} models · "
      f"{data.n_benchmarks} benchmarks · {lin.n_chains} lineage chains / {lin.n_deltas} steps")
print("axes:", AXIS)

400 draws/chain · 4280 obs · 758 models · 97 benchmarks · 36 lineage chains / 113 steps
axes: ['Hard math + science', 'Fluid/abstract + agentic', 'Easy knowledge']


### 1 · Seven chains share one basin; chain 7 sits **21.7 nats** below.
Chains 0-6 spread ±5.2 nats around their common mean.

In [2]:
lp = stats["logp"].values
gap = lp[ODD].mean() - lp[HEALTHY].mean(axis=1).mean()

fig = go.Figure()
for c in range(8):
    odd = c == ODD
    fig.add_trace(go.Violin(y=lp[c], x0=f"chain {c}", spanmode="hard", points=False,
                            name="chain 7 (second basin)" if odd else "chains 0-6 (shared basin)",
                            legendgroup="odd" if odd else "main",
                            showlegend=c in (0, ODD),
                            line_color=C["verm"] if odd else C["blue"],
                            fillcolor=C["verm"] if odd else C["sky"], opacity=0.55))
fig.add_trace(go.Scatter(x=[f"chain {c}" for c in range(8)], y=lp.mean(axis=1), mode="markers",
                         name="chain mean", marker=dict(color=C["dark"], symbol="diamond", size=9)))
fig.add_hline(y=float(lp[HEALTHY].mean()), line=dict(color=C["dark"], width=1, dash="dot"),
              annotation_text="mean of chains 0-6", annotation_position="top left")
fig.add_annotation(x="chain 7", y=float(lp[ODD].mean()), text=f"{gap:.1f} nats", showarrow=True,
                   arrowhead=2, ax=-72, ay=34, font=dict(color=C["verm"], size=12), arrowcolor=C["verm"])
fig.update_layout(title="Log posterior density per chain",
                  yaxis_title="logp", height=430, width=880,
                  legend=dict(orientation="h", y=1.06))
show(fig, "p1_logp_basins")

### 2 · Chains 0-6 are the same axes relabelled; chain 7 is a different solution.
Best-permutation loading correlation against chain 0: **0.996-1.000** for chains 1-6, **0.90 / 0.77 / 0.47** for chain 7.

In [3]:
fig = go.Figure(go.Heatmap(
    z=CORR, x=[f"{AXIS[k]}<br>(chain-0 axis {k + 1})" for k in range(3)],
    y=[f"chain {c}" for c in range(8)],
    zmin=0.4, zmax=1.0, colorscale=[[0, "#FFF7EC"], [0.5, C["orange"]], [1.0, C["blue"]]],
    colorbar=dict(title="corr", thickness=12),
    text=np.round(CORR, 3), texttemplate="%{text}", textfont=dict(size=12)))
fig.update_layout(title="Loading correlation with chain 0, after best axis permutation",
                  height=430, width=760, yaxis=dict(autorange="reversed"))
show(fig, "p2_axis_alignment")

### 3 · Chain 7 redraws the easy-vs-fluid boundary, mostly on thin new benchmarks.
**5 of the 12** biggest loading movers arrived in the 2026-07-27 refresh.

In [4]:
move = np.abs(A_odd - A_cons)[:, [EASY, FLUID]].sum(axis=1)
sel = np.argsort(-move)[:12][::-1]
labels = [f"<b>{benches[i]}</b>" if benches[i] in NEW else benches[i] for i in sel]
newmask = [benches[i] in NEW for i in sel]
symbols = ["diamond" if n else "circle" for n in newmask]

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=[f"{AXIS[EASY]} loading", f"{AXIS[FLUID]} loading"])
for col, k in enumerate([EASY, FLUID], 1):
    xs, ys = [], []
    for j, i in enumerate(sel):
        xs += [A_cons[i, k], A_odd[i, k], None]; ys += [labels[j], labels[j], None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=C["gray"], width=2),
                             name="loading shift", legendgroup="shift", showlegend=col == 1,
                             hoverinfo="skip"), row=1, col=col)
    for tag, vals, colr in [("consensus (chains 0-6)", A_cons[sel, k], C["blue"]),
                            ("chain 7", A_odd[sel, k], C["verm"])]:
        fig.add_trace(go.Scatter(x=vals, y=labels, mode="markers", name=tag, legendgroup=tag,
                                 showlegend=col == 1,
                                 marker=dict(color=colr, size=11, symbol=symbols,
                                             line=dict(color="white", width=1))),
                      row=1, col=col)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="new in 2026-07-27 refresh (bold)",
                         marker=dict(color=C["dark"], size=11, symbol="diamond")), row=1, col=1)
xmax = max(A_cons[sel][:, [EASY, FLUID]].max(), A_odd[sel][:, [EASY, FLUID]].max()) * 1.08
fig.update_xaxes(title_text="loading", range=[0, xmax])
fig.update_layout(title="Biggest loading movers, consensus → chain 7",
                  height=560, width=980, legend=dict(orientation="h", y=-0.13))
show(fig, "p3_boundary_movers")

### 4 · The per-family rate layer is empty: 36 vendors, one rate.
Spread across vendor medians is **0.010 / 0.045 / 0.022** logits/yr, 2-7% of the 0.49-0.65 mean posterior width.

In [5]:
rate = aligned("lin_rate", HEALTHY)              # (draws, 36, 3)
mu_pop = aligned("lin_rate_mu", HEALTHY)
med, lo, hi = (np.median(rate, axis=0), *np.quantile(rate, [0.05, 0.95], axis=0))
order = np.argsort(med[:, 0])
names = [lin.chain_names[i] for i in order]

fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.035,
                    subplot_titles=[f"{AXIS[k]}<br><span style='font-size:11px'>vendor sd "
                                    f"{med[:, k].std():.3f} · 90% CI {(hi[:, k] - lo[:, k]).mean():.2f}"
                                    "</span>" for k in range(3)])
for col, k in enumerate(range(3), 1):
    m = float(np.median(mu_pop[:, k]))
    fig.add_trace(go.Scatter(
        x=med[order, k], y=names, mode="markers", name="vendor rate (median, 90% CI)",
        legendgroup="v", showlegend=col == 1, marker=dict(color=C["blue"], size=6),
        error_x=dict(type="data", symmetric=False, array=hi[order, k] - med[order, k],
                     arrayminus=med[order, k] - lo[order, k], color=C["sky"], thickness=1.4, width=0)),
        row=1, col=col)
    fig.add_trace(go.Scatter(x=[m, m], y=[names[0], names[-1]], mode="lines",
                             name="population rate μ", legendgroup="mu", showlegend=col == 1,
                             line=dict(color=C["verm"], width=2)), row=1, col=col)
    fig.update_xaxes(title_text="logits / year", row=1, col=col)
fig.update_layout(title="Per-vendor improvement rate (healthy chains, axes aligned)",
                  height=780, width=1020, margin=dict(t=120), legend=dict(orientation="h", y=-0.07))
show(fig, "p4_vendor_rates")

### 5 · The hierarchy costs a plateau tax, not divergences.
Sampling 10,064 s vs 1,712 s for the no-BM baseline (5.9×), at **2 divergences / 16,000**; the cost is leapfrog steps.

In [6]:
steps = stats["n_steps"].mean("draw").values
divs = stats["diverging"].sum("draw").values

fig = go.Figure()
for tag, sel_c, colr in [("chains 0-6 (shared basin)", HEALTHY, C["blue"]),
                         ("chain 7 (second basin)", [ODD], C["verm"])]:
    fig.add_trace(go.Bar(x=[f"chain {c}" for c in sel_c], y=steps[sel_c], name=tag,
                         marker_color=colr, text=[f"{divs[c]} div" for c in sel_c],
                         textposition="outside", textfont=dict(size=11, color=C["dark"])))
fig.update_layout(title="Leapfrog steps per draw, and divergences",
                  yaxis_title="mean n_steps per draw", height=430, width=880,
                  yaxis_range=[0, steps.max() * 1.18], legend=dict(orientation="h", y=1.06))
show(fig, "p5_step_tax")

### 6 · Dropping chain 7 restores convergence.
Identified r̂ falls from **1.272 / 1.301 / 1.255** to **1.017 / 1.019 / 1.013** for eta / D / sigma_b.

In [7]:
r_all = mirt_identified_rhat(az.InferenceData(posterior=post), data)
r_drop = mirt_identified_rhat(az.InferenceData(posterior=post.isel(chain=HEALTHY)), data)
keys = ["eta_max_rhat", "D_max_rhat", "sigma_b_max_rhat"]
lab = ["eta (max)", "D (max)", "sigma_b (max)"]

fig = go.Figure()
for tag, r, colr in [("all 8 chains", r_all, C["verm"]), ("drop chain 7", r_drop, C["blue"])]:
    v = [r[k] for k in keys]
    fig.add_trace(go.Bar(x=lab, y=v, name=tag, marker_color=colr,
                         text=[f"{x:.3f}" for x in v], textposition="outside"))
fig.add_trace(go.Scatter(x=lab, y=[1.01] * 3, mode="lines", name="r̂ = 1.01 (convergence bar)",
                         line=dict(color=C["dark"], width=1, dash="dot")))
fig.update_layout(title="Identified r̂, with and without chain 7",
                  yaxis_title="r̂", yaxis_range=[1.0, 1.36], height=430, width=760,
                  legend=dict(orientation="h", y=1.06))
show(fig, "p6_rhat_drop7")

### 7 · What BM did measure: per-axis climb rates differ **3.5×**.
Fluid/agentic capability climbs 0.83 logits/yr, hard math+science 0.62, easy knowledge 0.23.

In [8]:
fig = go.Figure()
grid = np.linspace(mu_pop.min() - 0.15, mu_pop.max() + 0.1, 400)
for k, colr in zip(range(3), [C["blue"], C["orange"], C["green"]]):
    dens = gaussian_kde(mu_pop[:, k])(grid)
    m = float(np.median(mu_pop[:, k]))
    fig.add_trace(go.Scatter(x=grid, y=dens, mode="lines", fill="tozeroy", name=LEG[k],
                             line=dict(color=colr, width=2), opacity=0.75,
                             hovertemplate="%{x:.2f} logits/yr<extra></extra>"))
    fig.add_annotation(x=m, y=float(dens.max()), text=f"{m:.2f}", showarrow=False, yshift=10,
                       font=dict(size=12, color=colr))
ratio = np.median(mu_pop[:, FLUID]) / np.median(mu_pop[:, EASY])
fig.update_layout(title=f"Population climb rate per axis (ratio fastest / slowest = {ratio:.1f}×)",
                  xaxis_title="climb rate μ (logits / year)", yaxis_title="posterior density",
                  height=470, width=980, legend=dict(orientation="h", y=-0.22))
show(fig, "p7_axis_rates")

### Verdict
Per-family rate hierarchy retired 2026-07-27; shared per-axis drift with `dt` scaling kept.
Per-vendor rates stay readable post-hoc as `(psi_last − psi_founder) / T`, no hyperprior needed.
The second mode is the known easy-vs-fluid (axis 2-3) boundary, not the hyperprior.
The coverage caveat applies to any rate quoted: a chain's rate is measured only where its nodes are.